**Intelligent Vision Systems · Lab 6 · Camera Calibration**
Companion to **Lecture 6 — Camera Calibration**
Dr. Muhammad Kazim · Department of Intelligent Systems, University of Lahore

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kazimbalti/intelligent-vision-systems/blob/main/LAB/Lab_6/08-calibrate-your-camera.ipynb) &nbsp; [Lecture 6 slides](https://kazimbalti.github.io/intelligent-vision-systems/lectures/06_Camera_Calibration.html#/calibrate-your-own-camera) &nbsp; [All Lab 6 notebooks](https://kazimbalti.github.io/intelligent-vision-systems/labs.html#lab-6-calibration)

# Lab 6H · Calibrate Your Own Camera (Raspberry Pi / USB)

_Lecture 6, Section 6: calibrate your own camera._

Time to calibrate a **real** camera: the Raspberry Pi camera on your bench, a USB webcam or your laptop camera.
Run this notebook **on the machine the camera is attached to** (the Pi 5, or your laptop), not on Colab.

**You need:** a printed 9 × 6 chessboard ([pattern.png](https://github.com/opencv/opencv/blob/4.x/doc/pattern.png), 10 × 7
squares) glued to something **flat and rigid**, and a ruler to measure one square.

The same steps are packaged as two scripts in this folder: `capture_calib.py` and `calibrate.py`.

In [ ]:
# Setup (only needed on Google Colab or a fresh environment).
# Uncomment the next line to install the packages this notebook uses:
# !pip install -q opencv-python numpy matplotlib

import os, urllib.request

# Fetch the lab files if they are not next to the notebook.
RAW = "https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/LAB/Lab_6"
for f in ['lab6_checks.py', 'capture_calib.py', 'calibrate.py']:
    if not os.path.exists(f):
        os.makedirs(os.path.dirname(f) or ".", exist_ok=True)
        urllib.request.urlretrieve(f"{RAW}/{f}", f)
print("data ready")

In [ ]:
import glob, os, time
import numpy as np
import matplotlib.pyplot as plt
import cv2
import lab6_checks as C

PATTERN = (9, 6)        # inner corners
SQUARE_MM = 24.0        # TODO: measure YOUR printed square
OUT = "my_calib"
os.makedirs(OUT, exist_ok=True)

## 1. Capture 20–30 views

Pick **one** of the two capture cells. Move the board between shots: all over the frame (**especially the corners**),
tilted ±30–45° in both directions, near and far. Hold it **still** for every shot.

**Raspberry Pi camera (Picamera2):**

In [ ]:
USE_PI = False   # set True on the Raspberry Pi
if USE_PI:
    from picamera2 import Picamera2
    picam2 = Picamera2()
    picam2.configure(picam2.create_still_configuration(main={"size": (1920, 1080), "format": "RGB888"}))
    picam2.start(); time.sleep(1)
    try:
        picam2.set_controls({"AfMode": 0, "LensPosition": 1.0})   # Camera Module 3: lock focus (~1 m). Ignore on v2/HQ.
    except Exception as e:
        print("no autofocus control:", e)
    for i in range(25):
        time.sleep(3)
        frame = picam2.capture_array()
        ok, _ = cv2.findChessboardCorners(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), PATTERN, cv2.CALIB_CB_FAST_CHECK)
        cv2.imwrite(f"{OUT}/img_{i:02d}.jpg", frame)
        print(i, "✔ board" if ok else "✘ no board: move it into view")
    picam2.stop()

**USB webcam / laptop camera (press SPACE to save a view, ESC to finish):**

In [ ]:
USE_USB = False   # set True to use cv2.VideoCapture
if USE_USB:
    cap = cv2.VideoCapture(0)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280); cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    n = len(glob.glob(f"{OUT}/*.jpg"))
    while True:
        ok, frame = cap.read()
        if not ok: break
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        found, corners = cv2.findChessboardCorners(gray, PATTERN, cv2.CALIB_CB_FAST_CHECK)
        vis = frame.copy()
        if found: cv2.drawChessboardCorners(vis, PATTERN, corners, found)
        cv2.putText(vis, f"saved: {n}   SPACE = save, ESC = quit", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.imshow("capture", vis)
        k = cv2.waitKey(1) & 0xFF
        if k == 27: break
        if k == 32 and found:
            cv2.imwrite(f"{OUT}/img_{n:02d}.jpg", frame); n += 1
    cap.release(); cv2.destroyAllWindows()
print(len(glob.glob(f"{OUT}/*.jpg")), "images in", OUT)

## 2. Calibrate

(No camera at hand? Set `OUT = "data/camera_cal"` and `PATTERN = (9, 6)` and use the road-camera images.)

In [ ]:
files = sorted(glob.glob(f"{OUT}/*.jpg"))
objp = C.board_points(*PATTERN, SQUARE_MM)
objpoints, imgpoints, used, size = [], [], [], None
for f in files:
    g = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
    if size is None: size = g.shape[::-1]
    if g.shape[::-1] != size:
        print("skipping (different size):", f); continue
    ok, c = C.find_corners(g, PATTERN)
    if ok:
        objpoints.append(objp); imgpoints.append(c); used.append(f)
print(f"{len(used)} / {len(files)} usable views, size {size}")

rms, K, D, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, size, None, None)
print("RMS", rms); print(K); print(D.ravel())

In [ ]:
C.check_own_calibration(rms, K, size, len(used))

## 3. Quality checks: per-view error and coverage

In [ ]:
errs = []
for o, i, r, t in zip(objpoints, imgpoints, rvecs, tvecs):
    p, _ = cv2.projectPoints(o, r, t, K, D); errs.append(np.sqrt(((i.reshape(-1, 2) - p.reshape(-1, 2)) ** 2).sum(1).mean()))
fig, ax = plt.subplots(1, 2, figsize=(18, 5))
ax[0].bar(range(len(errs)), errs, color="#28568a"); ax[0].set_title("per-view RMS (px)")
allc = np.vstack([i.reshape(-1, 2) for i in imgpoints])
ax[1].scatter(allc[:, 0], allc[:, 1], s=3); ax[1].set_xlim(0, size[0]); ax[1].set_ylim(size[1], 0); ax[1].set_aspect("equal")
ax[1].set_title("coverage: are the image corners covered?")
plt.show()
print("worst views:", [os.path.basename(used[k]) for k in np.argsort(errs)[::-1][:3]])

## 4. Undistort and save `camera.yaml`

In [ ]:
img = cv2.imread(used[0])
newK, roi = cv2.getOptimalNewCameraMatrix(K, D, size, 0)
und = cv2.undistort(img, K, D, None, newK)
fig, ax = plt.subplots(1, 2, figsize=(18, 6)); ax[0].imshow(img[:, :, ::-1]); ax[0].set_title("raw"); ax[1].imshow(und[:, :, ::-1]); ax[1].set_title("undistorted (alpha = 0)")
for a in ax: a.axis("off")
plt.show()

fs = cv2.FileStorage("camera.yaml", cv2.FILE_STORAGE_WRITE)
fs.write("image_width", size[0]); fs.write("image_height", size[1])
fs.write("camera_matrix", K); fs.write("distortion_coefficients", D); fs.write("rms", rms)
fs.write("square_size_mm", SQUARE_MM); fs.release()
print(open("camera.yaml").read())

## 5. Use it: measure a real distance with `solvePnP`

With K and D known, one view of the board gives its pose, so we can measure the camera-to-board distance in mm.
Check it with your ruler!

In [ ]:
k = 0
ok, rvec, tvec = cv2.solvePnP(objpoints[k], imgpoints[k], K, D)
print(f"board origin is {np.linalg.norm(tvec):.0f} mm from the camera in {os.path.basename(used[k])}")

## Deliverable

Submit `camera.yaml`, the per-view error and coverage plots, one raw/undistorted pair, and short answers:

1. Your RMS error, fx, fy, cx, cy, k1. Compare fx with the datasheet value (Lecture 6 calculator slide).
2. Re-calibrate at a **different resolution** (e.g. 640 × 480). How do K and D change? Which numbers scale and which do not?
3. Measure the camera–board distance with a ruler for one view and compare it with `solvePnP`. Error in %?
4. (Optional) Calibrate with ROS 2: `ros2 run camera_calibration cameracalibrator --size 9x6 --square 0.024 ...` and compare.